# Post-hoc Attribution Rules (PHAR) Extraction & Optimization

This notebook systematically processes time-series datasets to extract structured, human-readable rules (PHAR) from continuous feature attributions (SHAP/LIME). To ensure memory efficiency and scalability, datasets are processed sequentially in a transactional manner—each dataset is loaded, optimized, processed, and its artifacts are saved to disk immediately before moving to the next.


## 1. Environment Setup & Global Configuration
Definition of base paths (`BASE_PATH = "shared/explain-ts/ds"`), tracking directories (e.g., timestamped run logs for March 1, 2026), and strict typing imports.
ENV:
 conda install -c conda-forge shap
 conda install -c conda-forge ipywidgets
 pip install "tensorflow[and-cuda]"


In [1]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # 0 - first gpu, 1 - second, "0,1" - both gpu, first used, "-1" - none

os.environ['LD_LIBRARY_PATH'] = f"{sys.prefix}/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

import gc
import json
import pickle
import shutil
import time
import traceback
import warnings
from typing import Any
from typing import Dict, List, Optional, Tuple, Union
from pathlib import Path
import numpy as np
import optuna
import pandas as pd
import shap
import tensorflow as tf
from scipy.stats import percentileofscore
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Zeros
from tensorflow.keras.layers import ConvLSTM1D, Input, Reshape, Dropout, Flatten, Dense
from tensorflow.keras.models import Sequential, load_model

print(tf.config.list_physical_devices('GPU'))

# The target directory structure expected by the rest of the notebook
BASE_PATH = "shared/explain-ts/ds"
# BASE_PATH = "shared/UCI-Benchmark/ds"

UNI_DIR = os.path.join(BASE_PATH, "univariate")
MULTI_DIR = os.path.join(BASE_PATH, "multivariate")


# Load model

class SafeConvLSTM1D(ConvLSTM1D):
    def __init__(self, *args, **kwargs):
        kwargs.pop('time_major', None)
        super().__init__(*args, **kwargs)


class SafeGlorotUniform(tf.keras.initializers.GlorotUniform):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)
        super().__init__(**kwargs)


class SafeOrthogonal(tf.keras.initializers.Orthogonal):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)
        super().__init__(**kwargs)


class SafeZeros(tf.keras.initializers.Zeros):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)  # Zeros might occasionally throw it too
        super().__init__()


# Crucial step: map the standard Keras names to our Safe wrappers
CUSTOM_OBJECTS = {
    'GlorotUniform': SafeGlorotUniform,
    'Orthogonal': SafeOrthogonal,
    'Zeros': SafeZeros,
    'ConvLSTM1D': SafeConvLSTM1D,
    'SafeConvLSTM1D': SafeConvLSTM1D
}


# --- Robust Loader ---
def load_benchmark_model(dataset_path: str, input_shape: tuple, num_classes: int) -> tf.keras.Model:
    h5_path = os.path.join(dataset_path, 'model.h5')
    tf_dir = os.path.join(dataset_path, 'model_tf/1')

    # 1. Standard load if healthy H5 exists
    if os.path.exists(h5_path):
        # We MUST pass CUSTOM_OBJECTS here to intercept 'dtype' during from_config()
        return load_model(h5_path, custom_objects=CUSTOM_OBJECTS, compile=False)

    # 2. Repair & Repack via Checkpoint Injection
    if os.path.isdir(tf_dir):
        print(f"Repacking legacy model for {os.path.basename(dataset_path)}...")

        # Build identical architecture using Safe layers to avoid initialization errors
        model = Sequential([
            Input(shape=input_shape),
            Reshape((*input_shape, 1), name='reshape'),
            SafeConvLSTM1D(64, kernel_size=3, padding='same', return_sequences=True, name='conv_lstm1d'),
            SafeConvLSTM1D(32, kernel_size=3, padding='same', return_sequences=True, name='conv_lstm1d_1'),
            Dropout(0.2, name='dropout'),
            Flatten(name='embedding'),
            Dense(100, activation='relu', name='dense'),
            Dense(num_classes, activation='softmax', name='dense_1')
        ])

        ckpt_prefix = os.path.join(tf_dir, 'variables', 'variables')

        try:
            checkpoint = tf.train.Checkpoint(model=model)
            checkpoint.restore(ckpt_prefix).expect_partial()
        except Exception as e:
            print(f"Checkpoint restore warning: {e}. Trying native Keras load_weights...")
            model.load_weights(ckpt_prefix)

        # Save healthy version for future runs
        model.save(h5_path)
        print("Successfully repacked to clean model.h5!")
        return model

    raise FileNotFoundError(f"No model artifacts found in {dataset_path}")


2026-03-02 11:44:00.597058: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Dataset Auditing & Explainer Availability
A fast, lightweight pass over the dataset registry to identify which datasets possess the required SHAP or LIME artifacts. Only fully validated datasets are queued for the main extraction loop.


In [2]:
def audit_datasets(categories_paths: Dict[str, str]) -> List[str]:
    """
    Iterates over all datasets to ensure they contain the required test data,
    a loadable Keras model, and at least one continuous explainer (SHAP or LIME).
    Fails fast if critical artifacts or both explainers are missing.
    Clears Keras session continuously to prevent OOM errors.

    Returns:
        List of absolute paths to fully verified datasets ready for PHAR extraction.
    """
    verified_datasets = []

    print("Starting Dataset Auditing & Explainer Availability Check...\n")

    for category, cat_path in categories_paths.items():
        if not os.path.exists(cat_path):
            print(f"Skipping {category}: Directory not found at {cat_path}")
            continue

        for ds_name in sorted(os.listdir(cat_path)):
            ds_path = os.path.join(cat_path, ds_name)
            if not os.path.isdir(ds_path):
                continue

            # 1. Check core data existence
            train_x_path = os.path.join(ds_path, 'trainX.pickle')
            train_y_path = os.path.join(ds_path, 'trainy.pickle')
            test_x_path = os.path.join(ds_path, 'testX.pickle')
            test_y_path = os.path.join(ds_path, 'testy.pickle')

            assert os.path.exists(train_x_path), f"FAIL FAST: Missing trainX.pickle in {ds_name}"
            assert os.path.exists(train_y_path), f"FAIL FAST: Missing trainy.pickle in {ds_name}"
            assert os.path.exists(test_x_path), f"FAIL FAST: Missing testX.pickle in {ds_name}"
            assert os.path.exists(test_y_path), f"FAIL FAST: Missing testy.pickle in {ds_name}"

            # 2. Check explainer existence
            shap_path = os.path.join(ds_path, 'svts.pickle')
            lime_path = os.path.join(ds_path, 'lvts.pickle')

            has_shap = os.path.exists(shap_path)
            has_lime = os.path.exists(lime_path)

            if not has_shap and not has_lime:
                raise FileNotFoundError(f"FAIL FAST: No SHAP or LIME artifacts found for {ds_name}!")
            elif not has_shap or not has_lime:
                missing = "SHAP" if not has_shap else "LIME"
                print(f"WARN: [{ds_name}] is missing {missing} explanations. Proceeding with available explainer.")

            # 3. Verify data loading & dimensions
            with open(test_x_path, 'rb') as f:
                testX = pickle.load(f)
            with open(test_y_path, 'rb') as f:
                testy = pickle.load(f)

            input_dim = testX.shape[1:]
            num_classes = testy.shape[1] if len(testy.shape) > 1 else len(np.unique(testy))

            # 4. Verify model loading
            try:
                model = load_benchmark_model(ds_path, input_shape=input_dim, num_classes=num_classes)
            except Exception as e:
                raise RuntimeError(f"FAIL FAST: Could not load model for {ds_name}. Error: {e}")

            # 5. Strict memory cleanup to prevent OOM in loop
            del model
            del testX
            del testy
            tf.keras.backend.clear_session()
            gc.collect()

            verified_datasets.append(ds_path)

    print(f"\nAudit complete. Successfully verified {len(verified_datasets)} datasets.")
    return verified_datasets


In [7]:
categories_to_audit = {
    "univariate": UNI_DIR,
    "multivariate": MULTI_DIR
}

verified_dataset_paths = audit_datasets(categories_to_audit)

Starting Dataset Auditing & Explainer Availability Check...



/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1772395321.624973    1243 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20266 MB memory:  -> device: 0, name: NVIDIA RTX A5500, pci bus id: 0000:51:00.0, compute capability: 8.6
I0000 00:00:1772395321.625450    1243 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22450 MB memory:  -> device: 1, name: NVIDIA RTX A5500, pci bus id: 0000:9c:00.0, compute capability: 8.6
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Inpu

WARN: [FaceDetection] is missing LIME explanations. Proceeding with available explainer.

Audit complete. Successfully verified 103 datasets.


In [3]:
verified_dataset_paths = ['shared/explain-ts/ds/univariate/Adiac',
                          'shared/explain-ts/ds/univariate/BME',
                          'shared/explain-ts/ds/univariate/Beef',
                          'shared/explain-ts/ds/univariate/BeetleFly',
                          'shared/explain-ts/ds/univariate/BirdChicken',
                          'shared/explain-ts/ds/univariate/CBF',
                          'shared/explain-ts/ds/univariate/Chinatown',
                          'shared/explain-ts/ds/univariate/Coffee',
                          'shared/explain-ts/ds/univariate/Computers',
                          'shared/explain-ts/ds/univariate/CricketX',
                          'shared/explain-ts/ds/univariate/CricketY',
                          'shared/explain-ts/ds/univariate/CricketZ',
                          'shared/explain-ts/ds/univariate/Crop',
                          'shared/explain-ts/ds/univariate/DiatomSizeReduction',
                          'shared/explain-ts/ds/univariate/DistalPhalanxOutlineAgeGroup',
                          'shared/explain-ts/ds/univariate/DistalPhalanxOutlineCorrect',
                          'shared/explain-ts/ds/univariate/DistalPhalanxTW',
                          'shared/explain-ts/ds/univariate/DodgerLoopDay',
                          'shared/explain-ts/ds/univariate/DodgerLoopGame',
                          'shared/explain-ts/ds/univariate/DodgerLoopWeekend',
                          'shared/explain-ts/ds/univariate/ECG200',
                          'shared/explain-ts/ds/univariate/ECG5000',
                          'shared/explain-ts/ds/univariate/ECGFiveDays',
                          'shared/explain-ts/ds/univariate/Earthquakes',
                          'shared/explain-ts/ds/univariate/ElectricDevices',
                          'shared/explain-ts/ds/univariate/FaceFour',
                          'shared/explain-ts/ds/univariate/FiftyWords',
                          'shared/explain-ts/ds/univariate/FordA',
                          'shared/explain-ts/ds/univariate/FordB',
                          'shared/explain-ts/ds/univariate/FreezerRegularTrain',
                          'shared/explain-ts/ds/univariate/FreezerSmallTrain',
                          'shared/explain-ts/ds/univariate/Fungi',
                          'shared/explain-ts/ds/univariate/GunPoint',
                          'shared/explain-ts/ds/univariate/GunPointAgeSpan',
                          'shared/explain-ts/ds/univariate/GunPointMaleVersusFemale',
                          'shared/explain-ts/ds/univariate/GunPointOldVersusYoung',
                          'shared/explain-ts/ds/univariate/Herring',
                          'shared/explain-ts/ds/univariate/InsectWingbeatSound',
                          'shared/explain-ts/ds/univariate/ItalyPowerDemand',
                          'shared/explain-ts/ds/univariate/LargeKitchenAppliances',
                          'shared/explain-ts/ds/univariate/Lightning2',
                          'shared/explain-ts/ds/univariate/Lightning7',
                          'shared/explain-ts/ds/univariate/Meat',
                          'shared/explain-ts/ds/univariate/MedicalImages',
                          'shared/explain-ts/ds/univariate/MiddlePhalanxOutlineAgeGroup',
                          'shared/explain-ts/ds/univariate/MiddlePhalanxOutlineCorrect',
                          'shared/explain-ts/ds/univariate/MiddlePhalanxTW',
                          'shared/explain-ts/ds/univariate/MoteStrain',
                          'shared/explain-ts/ds/univariate/OSULeaf',
                          'shared/explain-ts/ds/univariate/OliveOil',
                          'shared/explain-ts/ds/univariate/PhalangesOutlinesCorrect',
                          'shared/explain-ts/ds/univariate/Plane',
                          'shared/explain-ts/ds/univariate/PowerCons',
                          'shared/explain-ts/ds/univariate/ProximalPhalanxOutlineAgeGroup',
                          'shared/explain-ts/ds/univariate/ProximalPhalanxOutlineCorrect',
                          'shared/explain-ts/ds/univariate/ProximalPhalanxTW',
                          'shared/explain-ts/ds/univariate/RefrigerationDevices',
                          'shared/explain-ts/ds/univariate/ScreenType',
                          'shared/explain-ts/ds/univariate/ShapeletSim',
                          'shared/explain-ts/ds/univariate/ShapesAll',
                          'shared/explain-ts/ds/univariate/SmallKitchenAppliances',
                          'shared/explain-ts/ds/univariate/SmoothSubspace',
                          'shared/explain-ts/ds/univariate/SonyAIBORobotSurface1',
                          'shared/explain-ts/ds/univariate/SonyAIBORobotSurface2',
                          'shared/explain-ts/ds/univariate/Strawberry',
                          'shared/explain-ts/ds/univariate/SwedishLeaf',
                          'shared/explain-ts/ds/univariate/Symbols',
                          'shared/explain-ts/ds/univariate/SyntheticControl',
                          'shared/explain-ts/ds/univariate/ToeSegmentation2',
                          'shared/explain-ts/ds/univariate/Trace',
                          'shared/explain-ts/ds/univariate/TwoLeadECG',
                          'shared/explain-ts/ds/univariate/TwoPatterns',
                          'shared/explain-ts/ds/univariate/UMD',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryAll',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryX',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryY',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryZ',
                          'shared/explain-ts/ds/univariate/Wafer',
                          'shared/explain-ts/ds/univariate/Wine',
                          'shared/explain-ts/ds/univariate/WordSynonyms',
                          'shared/explain-ts/ds/univariate/Worms',
                          'shared/explain-ts/ds/univariate/WormsTwoClass',
                          'shared/explain-ts/ds/univariate/Yoga',
                          'shared/explain-ts/ds/multivariate/ArticularyWordRecognition',
                          'shared/explain-ts/ds/multivariate/AtrialFibrillation',
                          'shared/explain-ts/ds/multivariate/BasicMotions',
                          'shared/explain-ts/ds/multivariate/Cricket',
                          'shared/explain-ts/ds/multivariate/ERing',
                          'shared/explain-ts/ds/multivariate/Epilepsy',
                          'shared/explain-ts/ds/multivariate/EthanolConcentration',
                          'shared/explain-ts/ds/multivariate/FaceDetection',
                          'shared/explain-ts/ds/multivariate/FingerMovements',
                          'shared/explain-ts/ds/multivariate/HandMovementDirection',
                          'shared/explain-ts/ds/multivariate/Handwriting',
                          'shared/explain-ts/ds/multivariate/Heartbeat',
                          'shared/explain-ts/ds/multivariate/LSST',
                          'shared/explain-ts/ds/multivariate/Libras',
                          'shared/explain-ts/ds/multivariate/NATOPS',
                          'shared/explain-ts/ds/multivariate/PenDigits',
                          'shared/explain-ts/ds/multivariate/RacketSports',
                          'shared/explain-ts/ds/multivariate/SelfRegulationSCP1',
                          'shared/explain-ts/ds/multivariate/SelfRegulationSCP2',
                          'shared/explain-ts/ds/multivariate/UWaveGestureLibrary']

## 3. Core Classes: 3D-Aware Rule Generator
Implementation of the `GroundTruthRuleGenerator` adapted natively for 3D time-series formats `(n_samples, n_timesteps, n_variables)`. This includes overriding the perturbation mechanisms to handle temporal dimensions and abstracting the prediction logic for Keras `ConvLSTM-based` architectures.


In [4]:
def format_explanations_to_4d(explanations: Any, X_shape: tuple, num_classes: int) -> Tuple[np.ndarray, bool]:
    """
    Helper to ensure explanations are strictly shaped as (N, C, T, V).
    ExplainTS SHAP might be stored as a list of arrays or (N, T, V).

    Returns:
        A tuple (formatted_array, success_flag).
        success_flag is False if the array consists entirely of NaNs.
    """
    N, T, V = X_shape
    formatted_array = None

    if isinstance(explanations, list) and len(explanations) == num_classes:
        # e.g. List of C arrays, each (N, T, V)
        formatted_array = np.stack(explanations, axis=1)
    elif isinstance(explanations, np.ndarray):
        if explanations.ndim == 3:  # (N, T, V) for binary
            # Duplicate across classes for demonstration if missing class dim
            formatted_array = np.stack([explanations] * num_classes, axis=1)
        elif explanations.ndim == 4:
            formatted_array = explanations

    if formatted_array is None:
        raise ValueError(f"Unrecognized explanation shape/type: {type(explanations)}")

    # Check if the entire array consists of NaNs
    if np.isnan(formatted_array).all():
        print("WARN: Formatted explanation array contains ONLY NaN values.")
        return formatted_array, False

    return formatted_array, True


def get_stratified_pool(
        indices: np.ndarray,
        X: np.ndarray,
        expl: np.ndarray,
        y: np.ndarray,
        pool_fraction: float = 0.1,
        random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Safely extracts a stratified subset from the data based on a fraction.
    Bypasses sklearn's limitation with singleton classes and guarantees
    mathematical bounds for sample size.
    """
    total_samples = len(y)
    unique_classes, counts = np.unique(y, return_counts=True)
    num_classes = len(unique_classes)

    # Calculate target pool size based on fraction
    calculated_size = int(total_samples * pool_fraction)

    # Guard 1: Ensure enough samples to represent at least one of each class
    pool_size = max(calculated_size, num_classes)

    # Guard 2: Cap at the maximum available samples
    pool_size = min(pool_size, total_samples)

    # 1. Isolate singletons
    singleton_classes = unique_classes[counts == 1]
    singleton_mask = np.isin(y, singleton_classes)
    multiple_mask = ~singleton_mask

    indices_single = indices[singleton_mask]
    X_single = X[singleton_mask]
    expl_single = expl[singleton_mask]
    y_single = y[singleton_mask]

    remaining_size = pool_size - len(indices_single)

    # 2. Sample the rest of the data
    if remaining_size > 0 and multiple_mask.sum() > 0:
        # Guard 3: Do not request more samples than available in the non-singleton subset
        remaining_size = min(remaining_size, int(multiple_mask.sum()))

        try:
            indices_rest, _, X_rest, _, expl_rest, _, y_rest, _ = train_test_split(
                indices[multiple_mask],
                X[multiple_mask],
                expl[multiple_mask],
                y[multiple_mask],
                train_size=remaining_size,
                random_state=random_state,
                stratify=y[multiple_mask]
            )
        except ValueError as e:
            print(f"WARN: Stratification failed ({e}). Falling back to unstratified split.")
            indices_rest, _, X_rest, _, expl_rest, _, y_rest, _ = train_test_split(
                indices[multiple_mask],
                X[multiple_mask],
                expl[multiple_mask],
                y[multiple_mask],
                train_size=remaining_size,
                random_state=random_state,
                stratify=None
            )

        indices_pool = np.concatenate([indices_single, indices_rest])
        X_pool = np.concatenate([X_single, X_rest])
        expl_pool = np.concatenate([expl_single, expl_rest])
        y_pool = np.concatenate([y_single, y_rest])
    else:
        # If singletons exceed or match the requested pool size, just slice them
        indices_pool = indices_single[:pool_size]
        X_pool = X_single[:pool_size]
        expl_pool = expl_single[:pool_size]
        y_pool = y_single[:pool_size]

    return indices_pool, X_pool, expl_pool, y_pool




In [5]:
class PHARRuleGenerator(BaseEstimator, TransformerMixin):
    def __init__(self,
                 model: Any,
                 threshold_percentile: float = 40.0,
                 use_global_importance: bool = False,
                 perturb_sigma: float = 1.0,
                 perturbation_samples_count: int = 10_000,
                 min_selected_features: int = 1,
                 topk_fallback: int = 0,
                 cache_file: Optional[Union[Path, str]] = None, ):

        self.model = model
        self.threshold_percentile = float(threshold_percentile)
        self.use_global_importance = use_global_importance
        self.perturb_sigma = perturb_sigma
        self.perturbation_samples_count = perturbation_samples_count
        self.min_selected_features = int(min_selected_features)
        self.topk_fallback = int(topk_fallback)
        self.cache_file = cache_file
        self.cache_interval = 10

        self.n_timesteps = 0
        self.n_variables = 0
        self.n_classes = 0
        self.feature_names = []
        self.feature_coords = []

        self.class_thresholds = {}
        self.class_all_abs_explanations = {}
        self.class_abs_explanations_per_feature = {}
        self.feature_stats = {}

    def fit(self, X_train: np.ndarray, expl_train: np.ndarray) -> "PHARRuleGenerator":
        assert X_train.ndim == 3, f"X_train must be 3D (N, T, V), got {X_train.ndim}D"
        assert expl_train.ndim == 4, f"expl_train must be 4D (N, C, T, V), got {expl_train.ndim}D"

        n_samples, self.n_timesteps, self.n_variables = X_train.shape
        self.n_classes = expl_train.shape[1]

        for t in range(self.n_timesteps):
            for v in range(self.n_variables):
                if self.n_variables == 1:
                    self.feature_names.append(f"feature_{t}")
                else:
                    self.feature_names.append(f"var_{v}_ts_{t}")
                self.feature_coords.append((t, v))

        for class_idx in range(self.n_classes):
            sliced_expl = expl_train[:, class_idx, :, :]
            sliced_flat = sliced_expl.reshape(n_samples, -1)

            self.class_thresholds[class_idx] = {
                f_name: np.percentile(np.abs(sliced_flat[:, i]), self.threshold_percentile)
                for i, f_name in enumerate(self.feature_names)
            }

            self.class_all_abs_explanations[class_idx] = np.abs(sliced_flat).ravel()
            self.class_abs_explanations_per_feature[class_idx] = {
                f_name: np.abs(sliced_flat[:, i])
                for i, f_name in enumerate(self.feature_names)
            }

        X_flat = X_train.reshape(n_samples, -1)
        self.feature_stats = {
            f_name: {
                "mean": X_flat[:, i].mean(),
                "std": X_flat[:, i].std(),
                "min": X_flat[:, i].min(),
                "max": X_flat[:, i].max()
            }
            for i, f_name in enumerate(self.feature_names)
        }
        return self

    def transform(self, X_test: np.ndarray, expl_test: np.ndarray, original_indices: Optional[List[int]] = None) -> \
            List[Dict]:
        y_pred_proba = self.model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred_proba, axis=1)

        if original_indices is None:
            original_indices = list(range(X_test.shape[0]))

        # Try loading cache if provided
        if self.cache_file and os.path.exists(self.cache_file):
            with open(self.cache_file, 'rb') as f:
                rules = pickle.load(f)
            start_idx = len(rules)
            print(f"Resuming from cached index: {start_idx}")
        else:
            print("No cached data")
            rules = []
            start_idx = 0

        for idx in range(start_idx, X_test.shape[0]):
            start_time = time.time()
            instance = X_test[idx:idx + 1]
            original_prediction = y_pred_classes[idx]
            real_index = original_indices[idx]

            weights_for_pred = expl_test[idx, original_prediction, :, :].ravel()
            selected_features = []

            for i, f_name in enumerate(self.feature_names):
                abs_weight = abs(weights_for_pred[i])
                if self.use_global_importance:
                    exp_global_percentile = percentileofscore(self.class_all_abs_explanations[original_prediction],
                                                              abs_weight)
                    exceeds = exp_global_percentile >= self.threshold_percentile
                else:
                    exceeds = abs_weight >= self.class_thresholds[original_prediction][f_name]

                if exceeds:
                    f_stats = self.feature_stats[f_name]
                    selected_features.append((i, f_name, f_stats["min"], f_stats["max"], f_stats["std"]))

            if len(selected_features) < self.min_selected_features:
                print(f"WARN: Rule {idx} has less than {self.min_selected_features} selected features. ")
                if self.topk_fallback > 0:
                    top_idx = np.argsort(np.abs(weights_for_pred))[::-1]
                    top_idx = np.argsort(np.abs(weights_for_pred))[::-1]
                    used = {f_name for (_, f_name, *_) in selected_features}
                    added = 0
                    need = max(self.topk_fallback, self.min_selected_features - len(selected_features))

                    for fi in top_idx:
                        f_name = self.feature_names[fi]
                        if f_name not in used:
                            f_stats = self.feature_stats[f_name]
                            selected_features.append((fi, f_name, f_stats["min"], f_stats["max"], f_stats["std"]))
                            used.add(f_name)
                            added += 1
                            if added >= need:
                                break

            rule = {}
            confidence = 0.0
            coverage = 0.0

            if selected_features:

                perturbed_samples = []
                perturbed_metadata = []

                for _ in range(self.perturbation_samples_count):
                    p_sample = instance.copy()
                    meta_for_this_sample = []

                    for (fi, f_name, f_min, f_max, f_std) in selected_features:
                        t, v = self.feature_coords[fi]
                        orig_val = instance[0, t, v]
                        random_val = np.random.uniform(orig_val - self.perturb_sigma * f_std,
                                                       orig_val + self.perturb_sigma * f_std)
                        p_sample[0, t, v] = random_val
                        meta_for_this_sample.append((f_name, random_val))

                    perturbed_samples.append(p_sample[0])
                    perturbed_metadata.append(meta_for_this_sample)

                perturbed_array = np.array(perturbed_samples)
                p_preds = np.argmax(self.model.predict(perturbed_array, verbose=0), axis=1)

                pred_consistent_values = {}

                for sample_metadata, p_class in zip(perturbed_metadata, p_preds):
                    if p_class == original_prediction:
                        for f_name, val in sample_metadata:
                            pred_consistent_values.setdefault(f_name, []).append(val)

                for f_name, values in pred_consistent_values.items():
                    if len(values) == 1:
                        print(f"WARN: Rule {idx} has only 1 consistent value for feature {f_name}.")
                        val = values[0]
                        f_stats = self.feature_stats[f_name]
                        values.extend([
                            max(val - f_stats["std"], f_stats["min"]),
                            min(val + f_stats["std"], f_stats["max"])
                        ])

                    if len(values) > 1:
                        f_min, f_max = min(values), max(values)
                        rule[f_name] = [f">{f_min}", f"<={f_max}"]

                coverage, confidence = self._compute_coverage_and_confidence(rule, X_test, y_pred_classes,
                                                                             original_prediction)

            inference_time = time.time() - start_time

            rules.append({
                "index": int(real_index),
                "success": bool(rule),
                "prediction": int(original_prediction),
                "rule": rule,
                "confidence": confidence,
                "coverage": coverage,
                "exp_count": len(rule.keys()),
                "time_inference": inference_time,
                "method": "PHAR",
                "threshold_percentile": self.threshold_percentile,
                "use_global_importance": self.use_global_importance,
                "perturb_sigma": self.perturb_sigma,
                "perturbation_samples_count": self.perturbation_samples_count
            })

            # Save cache every 100 iterations
            if self.cache_file and ((idx + 1) % self.cache_interval == 0 or idx + 1 == X_test.shape[0]):
                with open(self.cache_file, 'wb') as f:
                    pickle.dump(rules, f)
                print(f"Checkpoint saved at index: {idx + 1}")

        return rules

    def _compute_coverage_and_confidence(self, rule: Dict[str, List[str]], X: np.ndarray,
                                         y_pred: np.ndarray, reference_class: int) -> Tuple[float, float]:
        if not rule:
            return 0.0, 0.0

        mask = np.ones(X.shape[0], dtype=bool)

        for f_name, interval in rule.items():
            lower_val = float(interval[0][1:])
            upper_val = float(interval[1][2:])

            fi = self.feature_names.index(f_name)
            t, v = self.feature_coords[fi]

            current_mask = (X[:, t, v] > lower_val) & (X[:, t, v] <= upper_val)
            mask = mask & current_mask

        coverage_value = mask.mean()
        if coverage_value == 0:
            return 0.0, 0.0

        covered_indices = np.where(mask)[0]
        confidence_value = np.mean(y_pred[covered_indices] == reference_class)
        return float(coverage_value), float(confidence_value)


def format_explanations_to_4d_strict(explanations: Any, expected_samples: int, num_classes: int, T: int,
                                     V: int) -> np.ndarray:
    """
    Helper to ensure explanations are strictly shaped as (N, C, T, V).
    Used internally by the fallback mechanism to standardize SHAP outputs.
    """
    if isinstance(explanations, list):
        if len(explanations) == num_classes:
            formatted_array = np.stack(explanations, axis=1)
        elif len(explanations) == 1 and num_classes == 2:
            # Binary classification edge case in some SHAP versions
            base_arr = explanations[0]
            formatted_array = np.stack([-base_arr, base_arr], axis=1)
        else:
            raise ValueError(f"Unexpected SHAP list length: {len(explanations)} for {num_classes} classes.")
    elif isinstance(explanations, np.ndarray):
        if explanations.ndim == 3:
            formatted_array = np.stack([explanations] * num_classes, axis=1)
        elif explanations.ndim == 4:
            formatted_array = explanations
        else:
            raise ValueError(f"Unexpected SHAP array ndim: {explanations.ndim}")
    else:
        raise ValueError(f"Unrecognized SHAP output type: {type(explanations)}")

    assert formatted_array.shape == (expected_samples, num_classes, T, V), \
        f"Shape mismatch. Expected {(expected_samples, num_classes, T, V)}, got {formatted_array.shape}"

    return formatted_array


def format_explanations_to_4d_strict(explanations: Any, expected_samples: int, num_classes: int, T: int,
                                     V: int) -> np.ndarray:
    """
    Helper to ensure explanations are strictly shaped as (N, C, T, V).
    Used internally by the fallback mechanism to standardize SHAP outputs.
    Automatically handles SHAP returning (N, T, V, C) by transposing the axes.
    """
    if isinstance(explanations, list):
        if len(explanations) == num_classes:
            formatted_array = np.stack(explanations, axis=1)
        elif len(explanations) == 1 and num_classes == 2:
            base_arr = explanations[0]
            formatted_array = np.stack([-base_arr, base_arr], axis=1)
        else:
            raise ValueError(f"Unexpected SHAP list length: {len(explanations)} for {num_classes} classes.")

    elif isinstance(explanations, np.ndarray):
        if explanations.ndim == 3:
            formatted_array = np.stack([explanations] * num_classes, axis=1)
        elif explanations.ndim == 4:
            # Check if SHAP returned (N, T, V, C) instead of (N, C, T, V)
            if explanations.shape == (expected_samples, T, V, num_classes):
                # Transpose from (0, 1, 2, 3) -> (0, 3, 1, 2)
                formatted_array = np.transpose(explanations, (0, 3, 1, 2))
            else:
                formatted_array = explanations
        else:
            raise ValueError(f"Unexpected SHAP array ndim: {explanations.ndim}")
    else:
        raise ValueError(f"Unrecognized SHAP output type: {type(explanations)}")

    assert formatted_array.shape == (expected_samples, num_classes, T, V), \
        f"Shape mismatch. Expected {(expected_samples, num_classes, T, V)}, got {formatted_array.shape}"

    return formatted_array


def compute_shap_in_batches(explainer: shap.GradientExplainer, X: np.ndarray, batch_size: int = 128,
                            cache_dir: str = None) -> Any:
    """
    Computes SHAP values in chunks to prevent OOM errors on GPU/RAM.
    Includes progress tracking, caching for resumption, and a fail-fast mechanism.
    Gracefully handles the structural warnings thrown by Keras inside tf.GradientTape.
    """
    n_samples = X.shape[0]
    shap_batches = []
    total_batches = (n_samples + batch_size - 1) // batch_size

    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*The structure of `inputs` doesn't match.*")

        for b_idx, i in enumerate(range(0, n_samples, batch_size)):
            cache_file = os.path.join(cache_dir, f"batch_{b_idx}.pickle") if cache_dir else None

            # 1. Resume mechanism: Check if this batch is already computed
            if cache_file and os.path.exists(cache_file):
                print(f"  -> Loading batch {b_idx + 1}/{total_batches} from cache...")
                with open(cache_file, 'rb') as f:
                    batch_vals = pickle.load(f)
            else:
                # 2. Compute mechanism: Process through the model
                print(f"  -> Computing batch {b_idx + 1}/{total_batches}...")
                X_batch = X[i: i + batch_size]
                batch_vals = explainer.shap_values(X_batch)

                # Fail-fast check ONLY on newly computed first batch
                if b_idx == 0:
                    if isinstance(batch_vals, list):
                        is_all_nan = all(np.isnan(c).all() for c in batch_vals)
                    else:
                        is_all_nan = np.isnan(batch_vals).all()

                    if is_all_nan:
                        raise RuntimeError("FAIL FAST: The first SHAP batch returned ONLY NaNs. Aborting early.")

                # Save newly computed batch to cache
                if cache_file:
                    with open(cache_file, 'wb') as f:
                        pickle.dump(batch_vals, f)

            shap_batches.append(batch_vals)

            gc.collect()
            tf.keras.backend.clear_session()

    if isinstance(shap_batches[0], list):
        num_classes = len(shap_batches[0])
        merged_list = []
        for c in range(num_classes):
            merged_class = np.concatenate([b[c] for b in shap_batches], axis=0)
            merged_list.append(merged_class)
        return merged_list
    else:
        return np.concatenate(shap_batches, axis=0)


def generate_and_save_fallback_shap(
        model: Any,
        X_train: np.ndarray,
        X_test: np.ndarray,
        num_classes: int,
        dataset_path: str,
        bg_samples: int = 50,
        batch_size: int = 32
) -> None:
    """
    Generates fallback SHAP explanations using GradientExplainer with batching and caching.
    Safely cleans up cache directories only upon full completion.
    """
    print(f"INFO: Initiating Gradient SHAP fallback for {os.path.basename(dataset_path)}...")

    N_tr, T, V = X_train.shape
    N_ts = X_test.shape[0]

    cache_dir_tr = os.path.join(dataset_path, '.cache_shap_tr')
    cache_dir_ts = os.path.join(dataset_path, '.cache_shap_ts')

    print(f"INFO: Clustering {N_tr} training samples into {bg_samples} background centroids...")
    X_train_2d = X_train.reshape(N_tr, T * V)
    n_clusters = min(bg_samples, N_tr)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(X_train_2d)
    background_3d = kmeans.cluster_centers_.reshape(n_clusters, T, V)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*The structure of `inputs` doesn't match.*")
        explainer = shap.GradientExplainer(model, background_3d)

    print(f"INFO: Processing SHAP values for TRAIN set...")
    shap_tr_raw = compute_shap_in_batches(explainer, X_train, batch_size=batch_size, cache_dir=cache_dir_tr)

    print(f"INFO: Processing SHAP values for TEST set...")
    shap_ts_raw = compute_shap_in_batches(explainer, X_test, batch_size=batch_size, cache_dir=cache_dir_ts)

    shap_tr_4d = format_explanations_to_4d_strict(shap_tr_raw, N_tr, num_classes, T, V)
    shap_ts_4d = format_explanations_to_4d_strict(shap_ts_raw, N_ts, num_classes, T, V)

    if np.isnan(shap_tr_4d).all() or np.isnan(shap_ts_4d).all():
        raise RuntimeError(f"FAIL FAST: Fallback Gradient SHAP returned ONLY NaNs for {dataset_path}.")

    if np.isnan(shap_ts_4d).any():
        print("WARN: Partial NaNs detected in fallback SHAP values. Downstream processing might be affected.")

    tr_path = os.path.join(dataset_path, 'svtr.pickle')
    ts_path = os.path.join(dataset_path, 'svts.pickle')

    print(f"INFO: Saving final artifacts to {tr_path} and {ts_path}...")
    with open(tr_path, 'wb') as f:
        pickle.dump(shap_tr_raw, f)

    with open(ts_path, 'wb') as f:
        pickle.dump(shap_ts_raw, f)

    # Safe cleanup ONLY after a successful write
    print("INFO: Cleaning up temporary cache directories...")
    if os.path.exists(cache_dir_tr):
        shutil.rmtree(cache_dir_tr)
    if os.path.exists(cache_dir_ts):
        shutil.rmtree(cache_dir_ts)

    print("INFO: Fallback generation complete and successfully saved.")

In [6]:
# test_dataset_path = next(p for p in verified_dataset_paths if "univariate" in p)  # "multivariate"
# test_dataset_path = next(p for p in verified_dataset_paths if "EthanolConcentration" in p)
test_dataset_path = next(p for p in verified_dataset_paths if "ArticularyWordRecognition" in p)
ds_name = os.path.basename(test_dataset_path)

print(f"--- Processing {ds_name} step-by-step ---")

# 1. Ładowanie danych Treningowych i Testowych
with open(os.path.join(test_dataset_path, 'trainX.pickle'), 'rb') as f:
    trainX = pickle.load(f)
with open(os.path.join(test_dataset_path, 'testX.pickle'), 'rb') as f:
    testX = pickle.load(f)
with open(os.path.join(test_dataset_path, 'testy.pickle'), 'rb') as f:
    testy = pickle.load(f)

print(f"testX shape: {testX.shape}")

# 2. Ładowanie modelu
input_dim = testX.shape[1:]
num_classes = testy.shape[1] if testy.ndim > 1 else len(np.unique(testy))
model = load_benchmark_model(test_dataset_path, input_shape=input_dim, num_classes=num_classes)

# 3. Ładowanie atrybucji SHAP (Trening i Test)
with open(os.path.join(test_dataset_path, 'svtr.pickle'), 'rb') as f:
    shap_tr_raw = pickle.load(f)
with open(os.path.join(test_dataset_path, 'svts.pickle'), 'rb') as f:
    shap_ts_raw = pickle.load(f)

shap_tr_4d, success_tr = format_explanations_to_4d(shap_tr_raw, trainX.shape, num_classes)
shap_ts_4d, success_ts = format_explanations_to_4d(shap_ts_raw, testX.shape, num_classes)


--- Processing ArticularyWordRecognition step-by-step ---
testX shape: (144, 144, 9)


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1772397507.568523    7071 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1038 MB memory:  -> device: 0, name: NVIDIA RTX A5500, pci bus id: 0000:51:00.0, compute capability: 8.6
I0000 00:00:1772397507.569341    7071 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22198 MB memory:  -> device: 1, name: NVIDIA RTX A5500, pci bus id: 0000:9c:00.0, compute capability: 8.6
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input

In [7]:
if not success_tr or not success_ts:
    print("WARN: Fallback SHAP values not found. Generating and saving...")
    generate_and_save_fallback_shap(model, trainX, testX, num_classes, test_dataset_path)

In [8]:
# 4. Losowanie stratyfikowanego poola ze zbioru testowego (z zachowaniem oryginalnych indeksów!)
y_true_classes = np.argmax(testy, axis=1) if testy.ndim > 1 else testy
print(y_true_classes)

[ 9 24 23 17  5 19 11 16  3  7 17 11 21  9 14  2 16  7 20 24 15  9  3  1
 20  9 13  8 23 10 18 16 10  7  1 12  7 10  1 19  5 12 20 18 12 21  5  2
 13  4 13 16  4 10 14  3  1  5 17  4 22 20 21 16 10  4 23  1  4  3  4 14
  4  2 11 17 11  2 21  4  8 18 17  0 23  9  5 21 12  8 14 14 21  1  0  6
  0  3 23  0 21 18 13 20  0 15 20 10  6 15 14  3 16 18 19  6 19  6  9 16
  3  6  6  6  2  0 15 11 21 23  7 11  5 24 12 19 20 16  2 24  5  3 21 14]


In [9]:
# Tworzymy wektor indeksów 0..N, który przepuścimy przez split
original_indices_array = np.arange(len(testX))

indices_pool, X_pool, expl_pool, y_pool = get_stratified_pool(
    original_indices_array, testX, shap_ts_4d, y_true_classes, pool_fraction=0.15
)

print(f"Wybrane indeksy próbek ze zbioru testowego: {indices_pool}")

Wybrane indeksy próbek ze zbioru testowego: [ 60 111   3 113 106  14   7  19   6   4  50  15  52 114 109  45 125  23
  41  17  87  28  21  95  37]


In [16]:
# 5. Generowanie reguł
generator = PHARRuleGenerator(
    model=model,
    threshold_percentile=90,
    perturbation_samples_count=1000,
    use_global_importance=False
)

# Krok FIT: Uczymy statystyki na CAŁYM zbiorze treningowym
print("Fitting global thresholds on TRAIN set...")
generator.fit(trainX, shap_tr_4d)

Fitting global thresholds on TRAIN set...


,model,"<Sequential n...l, built=True>"
,threshold_percentile,90.0
,use_global_importance,False
,perturb_sigma,1.0
,perturbation_samples_count,1000
,min_selected_features,1
,topk_fallback,0


In [17]:
print("Extracting rules on TEST pool...")
rules = generator.transform(X_pool[:10], expl_pool[:10], original_indices=list(indices_pool))

Extracting rules on TEST pool...


In [18]:
failed_rules = [r for r in rules if not r['success']]
print(f"Generated {len(rules) - len(failed_rules)} rules and {len(failed_rules)} failed.")

for r in rules[:2]:
    print("\n---------------------------")
    print(f"Original Index: {r['index']}")
    print(f"Predicted Class: {r['prediction']}")
    print(f"Success: {r['success']}")
    print(f"Coverage: {r['coverage']:.2f}, Confidence: {r['confidence']:.2f}")
    print(f"Inference Time: {r['time_inference']:.4f}s")
    print(
        f"Hyperparams: Perc={r['threshold_percentile']}, Global={r['use_global_importance']}, Sigma={r['perturb_sigma']}")
    print(f"exp_count: {r['exp_count']}")
    print("Rule Snippet:", list(r['rule'].items())[:3])

Generated 10 rules and 0 failed.

---------------------------
Original Index: 60
Predicted Class: 22
Success: True
Coverage: 0.10, Confidence: 1.00
Inference Time: 0.4502s
Hyperparams: Perc=90.0, Global=False, Sigma=1.0
exp_count: 29
Rule Snippet: [('var_0_ts_0', ['>0.5981944148492172', '<=2.846450991957985']), ('var_0_ts_1', ['>0.6448348562863486', '<=2.800828565552977']), ('var_0_ts_2', ['>0.6737704724325899', '<=2.7705276975155106'])]

---------------------------
Original Index: 111
Predicted Class: 3
Success: True
Coverage: 0.10, Confidence: 1.00
Inference Time: 0.4343s
Hyperparams: Perc=90.0, Global=False, Sigma=1.0
exp_count: 117
Rule Snippet: [('var_8_ts_0', ['>-2.9114656540583086', '<=-0.3620080634382763']), ('var_8_ts_1', ['>-3.1417099388201892', '<=-0.6495157204219355']), ('var_8_ts_2', ['>-3.1255045349843273', '<=-0.6666022377464604'])]


## 4. Hyperparameter Optimization Engine (Optuna)
Definition of the optimization objective. For a given dataset subset, Optuna searches for the optimal threshold and explainer base (SHAP vs. LIME) to maximize a harmonic mean of rule *Confidence* and *Coverage*.


In [6]:
class TimeAndTrialLimitCallback:
    """
    Custom Optuna callback to gracefully stop the study if the total time limit
    (including previous sessions) is exceeded, but strictly ensuring a minimum
    number of trials are completed.
    """

    def __init__(self, timeout_seconds: int, min_trials: int, prior_time_spent: float = 0.0):
        self.timeout_seconds = timeout_seconds
        self.min_trials = min_trials
        self.prior_time_spent = prior_time_spent
        self.session_start_time = time.time()

    def __call__(self, study: optuna.study.Study, trial: optuna.trial.FrozenTrial) -> None:
        # Calculate time spent in THIS specific run
        current_session_time = time.time() - self.session_start_time
        # Add it to the historical time from previous runs
        total_elapsed_time = self.prior_time_spent + current_session_time

        # Count only successfully completed trials
        completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])

        if total_elapsed_time > self.timeout_seconds and completed_trials >= self.min_trials:
            print(f"INFO: Stopping study. Total timeout reached ({total_elapsed_time:.1f}s) "
                  f"with {completed_trials} completed trials.")
            study.stop()


class PHARObjective:
    """
    Multi-objective optimization class for extracting PHAR rules.
    Optimizes for: (Maximize Confidence, Maximize Coverage, Minimize Sparsity)
    """

    def __init__(
            self,
            model: Any,
            X_train: np.ndarray,
            X_test_pool: np.ndarray,
            y_test_pool: np.ndarray,
            original_indices: List[int],
            explainers_train: Dict[str, np.ndarray],
            explainers_test: Dict[str, np.ndarray],
            study_name: str,
            jsonl_path: str
    ):
        self.model = model
        self.X_train = X_train
        self.X_test_pool = X_test_pool
        self.y_test_pool = y_test_pool
        self.original_indices = original_indices
        self.explainers_train = explainers_train
        self.explainers_test = explainers_test
        self.study_name = study_name
        self.jsonl_path = jsonl_path

        self.T = X_train.shape[1]
        self.V = X_train.shape[2]
        self.max_features = self.T * self.V

    def __call__(self, trial: optuna.trial.Trial) -> Tuple[float, float, float]:
        start_time = time.time()

        # 1. Hyperparameter suggestions
        available_methods = list(self.explainers_train.keys())
        explainer_choice = trial.suggest_categorical("explainer", available_methods)

        threshold_percentile = trial.suggest_float("threshold_percentile", 20.0, 99.0)
        perturb_sigma = trial.suggest_float("perturb_sigma", 0.1, 4.0)
        use_global_importance = trial.suggest_categorical("use_global_importance", [True, False])

        dynamic_perturbations = max(1000, min(5000, int(100 * self.V * np.sqrt(self.T))))
        trial.set_user_attr("perturbation_samples_count", dynamic_perturbations)

        # 2. Select the chosen explainer artifacts
        expl_tr = self.explainers_train[explainer_choice]
        expl_ts = self.explainers_test[explainer_choice]

        # 3. Rule Generation
        generator = PHARRuleGenerator(
            model=self.model,
            threshold_percentile=threshold_percentile,
            use_global_importance=use_global_importance,
            perturb_sigma=perturb_sigma,
            perturbation_samples_count=dynamic_perturbations
        )

        generator.fit(self.X_train, expl_tr)
        rules = generator.transform(self.X_test_pool, expl_ts, self.original_indices)

        # 4. Metric calculation
        total_samples = len(rules)
        if total_samples == 0:
            return 0.0, 0.0, float(self.max_features)

        confidences = [r['confidence'] for r in rules]
        coverages = [r['coverage'] for r in rules]
        sparsities = [r['exp_count'] if r['success'] else self.max_features for r in rules]

        # Averages for Optuna to optimize
        avg_confidence = float(np.mean(confidences))
        avg_coverage = float(np.mean(coverages))
        avg_sparsity = float(np.mean(sparsities))

        # 5. Comprehensive logging to JSONL
        record = {
            "trial_number": trial.number,
            "study_name": self.study_name,
            "params": trial.params,
            "dynamic_params": {
                "perturbation_samples_count": dynamic_perturbations
            },
            "metrics": {
                "confidence": {
                    "mean": avg_confidence, "std": float(np.std(confidences)),
                    "min": float(np.min(confidences)), "max": float(np.max(confidences)),
                    "median": float(np.median(confidences))
                },
                "coverage": {
                    "mean": avg_coverage, "std": float(np.std(coverages)),
                    "min": float(np.min(coverages)), "max": float(np.max(coverages)),
                    "median": float(np.median(coverages))
                },
                "sparsity": {
                    "mean": avg_sparsity, "std": float(np.std(sparsities)),
                    "min": float(np.min(sparsities)), "max": float(np.max(sparsities)),
                    "median": float(np.median(sparsities))
                }
            },
            "total_time": str(time.time() - start_time),
            "rules": rules,
        }

        with open(self.jsonl_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        del generator, rules, record, confidences, coverages, sparsities
        tf.keras.backend.clear_session()
        gc.collect()

        return avg_confidence, avg_coverage, avg_sparsity


def run_optimization_for_dataset(
        dataset_path: str,
        db_path: str = "sqlite:///.optuna_phar.sqlite3",
        pool_fraction: float = 0.1,
        timeout: int = 2 * 60 * 60,
        min_trials: int = 5,
        n_trials: int = 50,
        is_test_run: bool = False
) -> None:
    """
    Main pipeline entrypoint for a single dataset. Loads artifacts, extracts
    a stratified pool, sets up Optuna, and runs the multi-objective search.
    Handles NaN values in explainers gracefully.
    """
    base_ds_name = os.path.basename(dataset_path)

    # Configure test run overrides
    study_name = f"{base_ds_name}_test" if is_test_run else base_ds_name
    jsonl_filename = "phar_trials_log_test.jsonl" if is_test_run else "phar_trials_log.jsonl"
    jsonl_path = os.path.join(dataset_path, jsonl_filename)

    print(f"\n========== Starting Optimization: {study_name} ==========")

    # 1. Load basic data
    with open(os.path.join(dataset_path, 'trainX.pickle'), 'rb') as f:
        trainX = pickle.load(f)
    with open(os.path.join(dataset_path, 'testX.pickle'), 'rb') as f:
        testX = pickle.load(f)
    with open(os.path.join(dataset_path, 'testy.pickle'), 'rb') as f:
        testy = pickle.load(f)

    input_dim = testX.shape[1:]
    num_classes = testy.shape[1] if testy.ndim > 1 else len(np.unique(testy))
    model = load_benchmark_model(dataset_path, input_shape=input_dim, num_classes=num_classes)
    y_true_classes = np.argmax(testy, axis=1) if testy.ndim > 1 else testy

    # 2. Load available explainers & check for NaNs
    explainers_tr = {}
    explainers_ts = {}

    # --- SHAP Check ---
    shap_tr_path = os.path.join(dataset_path, 'svtr.pickle')
    shap_ts_path = os.path.join(dataset_path, 'svts.pickle')
    if os.path.exists(shap_tr_path) and os.path.exists(shap_ts_path):
        with open(shap_tr_path, 'rb') as f:
            tr_shap = pickle.load(f)
        with open(shap_ts_path, 'rb') as f:
            ts_shap = pickle.load(f)

        tr_shap_4d, tr_valid = format_explanations_to_4d(tr_shap, trainX.shape, num_classes)
        ts_shap_4d, ts_valid = format_explanations_to_4d(ts_shap, testX.shape, num_classes)

        if tr_valid and ts_valid:
            explainers_tr["SHAP"] = tr_shap_4d
            explainers_ts["SHAP"] = ts_shap_4d
        else:
            print(f"WARN: SHAP artifacts contain ONLY NaNs for {base_ds_name}. Excluding SHAP.")

    # --- LIME Check ---
    lime_tr_path = os.path.join(dataset_path, 'lvtr.pickle')
    lime_ts_path = os.path.join(dataset_path, 'lvts.pickle')
    if os.path.exists(lime_tr_path) and os.path.exists(lime_ts_path):
        with open(lime_tr_path, 'rb') as f:
            tr_lime = pickle.load(f)
        with open(lime_ts_path, 'rb') as f:
            ts_lime = pickle.load(f)

        tr_lime_4d, tr_valid = format_explanations_to_4d(tr_lime, trainX.shape, num_classes)
        ts_lime_4d, ts_valid = format_explanations_to_4d(ts_lime, testX.shape, num_classes)

        if tr_valid and ts_valid:
            explainers_tr["LIME"] = tr_lime_4d
            explainers_ts["LIME"] = ts_lime_4d
        else:
            print(f"WARN: LIME artifacts contain ONLY NaNs for {base_ds_name}. Excluding LIME.")

    # 3. Validate at least one explainer works
    if not explainers_tr:
        print(f"ERROR: No valid explainers (SHAP or LIME) found for {base_ds_name}. Skipping dataset entirely.")
        del model, trainX, testX, testy
        tf.keras.backend.clear_session()
        gc.collect()
        return

    # 4. Create stratified pool
    dummy_expl = list(explainers_ts.values())[0]
    original_indices_array = np.arange(len(testX))

    indices_pool, X_pool, _, y_pool = get_stratified_pool(
        original_indices_array, testX, dummy_expl, y_true_classes, pool_fraction=pool_fraction
    )

    pool_explainers_ts = {
        name: expl[indices_pool] for name, expl in explainers_ts.items()
    }

    # 5. Optuna Study setup
    sampler = optuna.samplers.TPESampler(n_startup_trials=int(n_trials / 5), seed=42)
    study = optuna.create_study(
        study_name=study_name,
        storage=db_path,
        directions=["maximize", "maximize", "minimize"],
        sampler=sampler,
        load_if_exists=True
    )

    # 1. Calculate historical metrics from SQLite
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    num_completed = len(completed_trials)

    # Sum the duration of all past trials that have finished
    prior_time_spent = sum(
        (t.datetime_complete - t.datetime_start).total_seconds()
        for t in study.trials
        if t.datetime_complete is not None and t.datetime_start is not None
    )

    remaining_trials = max(0, n_trials - num_completed)

    # 2. Smart Skip Logic
    if remaining_trials == 0:
        print(f"INFO: Study '{study_name}' already reached the target of {num_completed} trials. Skipping.")
    elif prior_time_spent > timeout and num_completed >= min_trials:
        print(f"INFO: Study '{study_name}' already exceeded the {timeout}s timeout "
              f"(spent {prior_time_spent:.1f}s) and has {num_completed} trials. Skipping.")
    else:
        # 3. Resume / Start Optimization
        print(f"INFO: Starting/Resuming study. Target: {remaining_trials} more trials. "
              f"Prior time spent: {prior_time_spent:.1f}s.")

        objective = PHARObjective(
            model=model,
            X_train=trainX,
            X_test_pool=X_pool,
            y_test_pool=y_pool,
            original_indices=list(indices_pool),
            explainers_train=explainers_tr,
            explainers_test=pool_explainers_ts,
            study_name=study_name,
            jsonl_path=jsonl_path
        )

        time_callback = TimeAndTrialLimitCallback(timeout_seconds=timeout, min_trials=min_trials,
                                                  prior_time_spent=prior_time_spent)

        # print(f"INFO: Running study for {n_trials} trials (Timeout: {timeout}s)...")
        study.optimize(
            objective,
            n_trials=n_trials,
            callbacks=[time_callback],
            gc_after_trial=True
        )

    del model, trainX, testX, testy, explainers_tr, explainers_ts
    tf.keras.backend.clear_session()
    gc.collect()


In [7]:
def extract_final_rules(dataset_path: str, is_test_run: bool = False) -> None:
    """
    Reads the optimization JSONL log, selects the best hyperparameter configuration
    based on a hierarchical heuristic (Confidence > Coverage > Sparsity).
    Generates and saves final PHAR rules for both TRAIN and TEST sets as .pickle
    files, formatted as a list of single-element lists for compatibility.
    Saves a lightweight metadata JSON.
    """
    base_ds_name = os.path.basename(dataset_path)

    # Define file paths based on run mode
    jsonl_filename = "phar_trials_log_test.jsonl" if is_test_run else "phar_trials_log.jsonl"
    jsonl_path = os.path.join(dataset_path, jsonl_filename)

    meta_filename = "phar_metadata_test.json" if is_test_run else "phar_metadata.json"
    meta_path = os.path.join(dataset_path, meta_filename)

    pvtr_filename = "pvtr_test.pickle" if is_test_run else "pvtr.pickle"
    pvts_filename = "pvts_test.pickle" if is_test_run else "pvts.pickle"
    pvtr_path = os.path.join(dataset_path, pvtr_filename)
    pvts_path = os.path.join(dataset_path, pvts_filename)

    print(f"\n========== Extracting Final Rules: {base_ds_name} ==========")

    # 1. Parse JSONL and select the best trial
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    # Hierarchical sorting: Maximize Confidence -> Maximize Coverage -> Minimize Sparsity
    best_record = sorted(
        records,
        key=lambda x: (
            x["metrics"]["confidence"]["mean"],
            x["metrics"]["coverage"]["mean"],
            -x["metrics"]["sparsity"]["mean"]
        ),
        reverse=True
    )[0]

    best_params = best_record["params"]
    best_dynamic = best_record["dynamic_params"]
    explainer_choice = best_params["explainer"]

    print(f"INFO: Selected Trial {best_record['trial_number']} using {explainer_choice}.")

    # 2. Load dataset
    with open(os.path.join(dataset_path, 'trainX.pickle'), 'rb') as f:
        trainX = pickle.load(f)
    with open(os.path.join(dataset_path, 'testX.pickle'), 'rb') as f:
        testX = pickle.load(f)
    with open(os.path.join(dataset_path, 'testy.pickle'), 'rb') as f:
        testy = pickle.load(f)

    if is_test_run:
        print("INFO: Test run active. Slicing sets to the first 10 samples.")
        trainX = trainX[:10]
        testX = testX[:10]
        testy = testy[:10]

    original_indices_tr = list(range(len(trainX)))
    original_indices_ts = list(range(len(testX)))

    input_dim = testX.shape[1:]
    num_classes = testy.shape[1] if testy.ndim > 1 else len(np.unique(testy))
    model = load_benchmark_model(dataset_path, input_shape=input_dim, num_classes=num_classes)

    # 3. Load ONLY the required explainer artifacts
    tr_raw_path = 'svtr.pickle' if explainer_choice == "SHAP" else 'lvtr.pickle'
    ts_raw_path = 'svts.pickle' if explainer_choice == "SHAP" else 'lvts.pickle'

    with open(os.path.join(dataset_path, tr_raw_path), 'rb') as f:
        raw_tr = pickle.load(f)
    with open(os.path.join(dataset_path, ts_raw_path), 'rb') as f:
        raw_ts = pickle.load(f)

    expl_tr, _ = format_explanations_to_4d(raw_tr, trainX.shape, num_classes)
    expl_ts, _ = format_explanations_to_4d(raw_ts, testX.shape, num_classes)

    if is_test_run:
        expl_tr = expl_tr[:10]
        expl_ts = expl_ts[:10]

    if is_test_run:
        perturbation_samples_count = 100
    else:
        perturbation_samples_count = min(3 * best_dynamic["perturbation_samples_count"], 10_000)

    # 4. Initialize Generator
    generator = PHARRuleGenerator(
        model=model,
        threshold_percentile=best_params["threshold_percentile"],
        use_global_importance=best_params["use_global_importance"],
        perturb_sigma=best_params["perturb_sigma"],
        perturbation_samples_count=perturbation_samples_count,
        cache_file=os.path.join(dataset_path, f"phar_cache_{explainer_choice}_tr.pickle")
    )

    print("INFO: Fitting global thresholds on TRAIN set...")
    generator.fit(trainX, expl_tr)

    # 5. Extract and Save TRAIN rules
    skip_train = False
    if os.path.exists(pvtr_path):
        try:
            with open(pvtr_path, 'rb') as f:
                existing_tr = pickle.load(f)
            if len(existing_tr) == len(trainX):
                print(f"INFO: Complete TRAIN rules already exist at {pvtr_filename}. Skipping extraction.")
                skip_train = True
            # else:
            #     print(f"WARN: Incomplete TRAIN rules found ({len(existing_tr)}/{len(trainX)}). Recomputing...")
        except Exception as e:
            # print(f"WARN: Corrupted TRAIN rules file ({e}). Recomputing...")
            pass

    if not skip_train:
        print(f"INFO: Extracting final rules for {len(trainX)} TRAIN samples...")
        rules_tr = generator.transform(trainX, expl_tr, original_indices=original_indices_tr)

        # Wrap each rule in a list for compatibility: [ [{...}], [{...}] ]
        formatted_rules_tr = [[r] for r in rules_tr]

        with open(pvtr_path, 'wb') as f:
            pickle.dump(formatted_rules_tr, f)
        print(f"SUCCESS: Train rules saved to {pvtr_filename}.")

        # Aggressive memory cleanup before processing test set
        del rules_tr, formatted_rules_tr
        gc.collect()

    # 6. Extract and Save TEST rules
    print(f"INFO: Extracting final rules for {len(testX)} TEST samples...")
    generator.cache_file = os.path.join(dataset_path, f"phar_cache_{explainer_choice}_ts.pickle")
    rules_ts = generator.transform(testX, expl_ts, original_indices=original_indices_ts)

    # Wrap each rule in a list for compatibility
    formatted_rules_ts = [[r] for r in rules_ts]

    with open(pvts_path, 'wb') as f:
        pickle.dump(formatted_rules_ts, f)
    print(f"SUCCESS: Test rules saved to {pvts_filename}.")

    del rules_ts, formatted_rules_ts
    gc.collect()

    # 7. Save Lightweight Metadata JSON
    metadata = {
        "dataset_name": base_ds_name,
        "is_test_run": is_test_run,
        "best_trial": best_record,
        "artifacts_generated": [pvtr_filename, pvts_filename]
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)
    print(f"SUCCESS: Metadata saved to {meta_filename}.")

    # Final cleanup
    del model, trainX, testX, testy, expl_tr, expl_ts, raw_tr, raw_ts
    tf.keras.backend.clear_session()
    gc.collect()


In [52]:
# --- Example Usage (Test Run Demonstration) ---
uni_demo_dataset_path = next(
    p for p in verified_dataset_paths if "univariate" in p)  # "multivariate" # verified_dataset_paths[0]

run_optimization_for_dataset(
    dataset_path=uni_demo_dataset_path,
    pool_fraction=0.05,
    timeout=300,
    min_trials=3,
    n_trials=5,
    is_test_run=True  # Important: Safely flags this as a test!
)


========== Starting Optimization: Adiac_test ==========


[I 2026-03-02 10:18:09,647] Using an existing study with name 'Adiac_test' instead of creating a new one.


INFO: Study 'Adiac_test' already reached the target of 5 trials. Skipping.


In [31]:
extract_final_rules(dataset_path=uni_demo_dataset_path, is_test_run=True)


========== Extracting Final Rules: Adiac ==========
INFO: Selected Trial 2 using SHAP.
INFO: Test run active. Slicing sets to the first 10 samples.
INFO: Fitting global thresholds on TRAIN set...
INFO: Extracting final rules for 10 TRAIN samples...


2026-03-01 22:20:22.308289: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_757', 4 bytes spill stores, 4 bytes spill loads



SUCCESS: Train rules saved to pvtr_test.pickle.
INFO: Extracting final rules for 10 TEST samples...
SUCCESS: Test rules saved to pvts_test.pickle.
SUCCESS: Metadata saved to phar_metadata_test.json.


In [26]:
# --- Example Usage (Test Run Demonstration) ---
multi_demo_dataset_path = next(
    p for p in verified_dataset_paths if "multivariate" in p)  # "univariate" # verified_dataset_paths[0]

run_optimization_for_dataset(
    dataset_path=multi_demo_dataset_path,
    pool_fraction=0.05,
    timeout=300,
    min_trials=3,
    n_trials=5,
    is_test_run=True  # Important: Safely flags this as a test!
)


========== Starting Optimization: ArticularyWordRecognition_test ==========


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-03-01 21:54:07,922] A new study created in RDB with name: ArticularyWordRecognition_test


INFO: Running study for 5 trials (Timeout: 300s)...


[I 2026-03-01 21:55:51,074] Trial 0 finished with values: [0.4596666666666666, 0.14880000000000002, 252.24] and parameters: {'explainer': 'LIME', 'threshold_percentile': 80.50758198498696, 'perturb_sigma': 2.434768088368443, 'use_global_importance': True}.
[I 2026-03-01 21:57:49,774] Trial 1 finished with values: [0.48034415584415585, 0.2464, 371.36] and parameters: {'explainer': 'LIME', 'threshold_percentile': 71.47693581028142, 'perturb_sigma': 2.8614830534045774, 'use_global_importance': False}.
[I 2026-03-01 22:01:35,024] Trial 2 finished with values: [1.0, 0.04, 775.44] and parameters: {'explainer': 'SHAP', 'threshold_percentile': 42.54592273728994, 'perturb_sigma': 0.8152775884283918, 'use_global_importance': False}.


INFO: Stopping study. Timeout reached (447.3s) with 3 trials completed.


In [28]:
extract_final_rules(dataset_path=multi_demo_dataset_path, is_test_run=True)


========== Extracting Final Rules: ArticularyWordRecognition ==========
INFO: Selected Trial 2 using SHAP.
INFO: Expected Metrics -> Conf: 1.0000, Cov: 0.0400
INFO: Test run active. Slicing test set to the first 10 samples.


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


INFO: Fitting global thresholds on TRAIN set...
INFO: Extracting final rules for 10 TEST samples...
SUCCESS: Final rules saved to phar_final_rules_test.json.


## 5. Main Extraction Pipeline (Per-Dataset Transaction)
The core execution loop. For each dataset in the verified queue, this block performs the following isolated steps:
1. **Load:** Fetch model, test sequences (`X`, `y`), and raw explanations.
2. **Subsample:** Create a stratified rule extraction pool from the test set.
3. **Optimize:** Run Optuna on the subsample to find the best configuration.
4. **Extract:** Generate final PHAR rules for the *entire* test set using `globalF`, `globalT`, and the `best` configurations.
5. **Serialize:** Instantly save the resulting `.pickle` arrays and JSON search histories to the standardized ExplainTS structure.
6. **Cleanup:** Explicitly clear memory and TensorFlow sessions to prevent OOM errors during long-running batch processing.


In [8]:
def process_datasets(
        paths: List[str],
        timeout: int,
        pool_fraction: float,
        is_test_run: bool,
        error_log_path: str,
        min_trials: int = 20,
        n_trials: int = 60,
        db_path: str = "sqlite:///.optuna_phar.sqlite3",
) -> None:
    """
    Executes the full PHAR extraction transaction (Optimization + Final Extraction)
    for a list of datasets. Includes skip-logic for already processed datasets
    and robust error handling to ensure continuous execution.
    """
    for dataset_path in paths:
        ds_name = os.path.basename(dataset_path)

        # 1. Skip Check (Transaction safety)
        target_filename = "pvts_test.pickle" if is_test_run else "pvts.pickle"
        if os.path.exists(os.path.join(dataset_path, target_filename)):
            print(f"\n--- SKIPPING {ds_name}: Target artifact '{target_filename}' already exists. ---")
            continue

        print(f"\n{'=' * 50}")
        print(f" STARTING TRANSACTION: {ds_name}")
        print(f"{'=' * 50}")

        try:
            # Step A: Hyperparameter Tuning
            run_optimization_for_dataset(
                dataset_path=dataset_path,
                db_path=db_path,
                pool_fraction=pool_fraction,
                timeout=timeout,
                min_trials=min_trials,
                n_trials=n_trials,
                is_test_run=is_test_run
            )

            # Step B: Final Rule Extraction
            extract_final_rules(
                dataset_path=dataset_path,
                is_test_run=is_test_run
            )

            print(f"\n>>> TRANSACTION SUCCESSFUL: {ds_name} <<<")

        except Exception as e:
            # Step C: Graceful Failure Handling
            error_msg = traceback.format_exc()
            print(f"\n!!! TRANSACTION FAILED: {ds_name} !!!")
            print(f"Error: {e}")
            print(f"Logging trace to {error_log_path} and continuing to next dataset...")

            with open(error_log_path, "a", encoding="utf-8") as f:
                f.write(f"Dataset: {ds_name}\n")
                f.write(f"Mode: {'TEST' if is_test_run else 'PROD'}\n")
                f.write(f"Exception: {str(e)}\n")
                f.write(f"Traceback:\n{error_msg}\n")
                f.write("-" * 60 + "\n")

        finally:
            # Step D: Hard Cleanup (Executed even if an error occurs)
            tf.keras.backend.clear_session()
            gc.collect()


In [13]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
IS_TEST_RUN = False
# 1. Filter datasets by category
# Assuming verified_dataset_paths is already populated from the Audit step
uni_paths = [p for p in verified_dataset_paths if "univariate" in p]
multi_paths = [p for p in verified_dataset_paths if "multivariate" in p]

print(f"Prepared {len(uni_paths)} univariate and {len(multi_paths)} multivariate datasets.")


Prepared 83 univariate and 20 multivariate datasets.


In [ ]:
# 2. Run Univariate Loop
# 1 Hour timeout (3600s), 15% stratified pool
print("\n" + "#" * 50)
print(" INITIATING UNIVARIATE PIPELINE")
print("#" * 50)

process_datasets(
    paths=uni_paths,
    timeout=60 * 60,
    pool_fraction=0.15,
    is_test_run=IS_TEST_RUN,
    error_log_path="failed_datasets_univariate.txt",
    min_trials=20,
    n_trials=60,
    db_path="sqlite:///.optuna_phar_uni.sqlite3",
)


##################################################
 INITIATING UNIVARIATE PIPELINE
##################################################

--- SKIPPING Adiac: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING BME: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING Beef: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING BeetleFly: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING BirdChicken: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING CBF: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING Chinatown: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING Coffee: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING Computers: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING CricketX: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING CricketY: Target artifact 'pvts.pickle' already exists. ---

--- SKIPPING CricketZ: Target artifact 'pvts.pickle' already ex

/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-03-03 09:25:26,915] Using an existing study with name 'FordB' instead of creating a new one.


INFO: Starting/Resuming study. Target: 44 more trials. Prior time spent: 3861.9s.
No cached data


[I 2026-03-03 09:29:28,079] Trial 17 finished with values: [1.0, 0.006024096385542167, 89.69277108433735] and parameters: {'explainer': 'SHAP', 'threshold_percentile': 84.56640089378664, 'perturb_sigma': 0.12943454533354393, 'use_global_importance': True}.


No cached data
WARN: Rule 56 has less than 1 selected features. 
WARN: Rule 77 has only 1 consistent value for feature feature_7.
WARN: Rule 77 has only 1 consistent value for feature feature_9.
WARN: Rule 77 has only 1 consistent value for feature feature_10.
WARN: Rule 77 has only 1 consistent value for feature feature_11.
WARN: Rule 77 has only 1 consistent value for feature feature_12.
WARN: Rule 77 has only 1 consistent value for feature feature_14.
WARN: Rule 77 has only 1 consistent value for feature feature_18.
WARN: Rule 77 has only 1 consistent value for feature feature_19.
WARN: Rule 77 has only 1 consistent value for feature feature_20.
WARN: Rule 77 has only 1 consistent value for feature feature_27.
WARN: Rule 77 has only 1 consistent value for feature feature_69.
WARN: Rule 77 has only 1 consistent value for feature feature_70.
WARN: Rule 77 has only 1 consistent value for feature feature_73.
WARN: Rule 77 has only 1 consistent value for feature feature_85.
WARN: Rule 77

...

No cached data


[I 2026-03-03 21:50:49,447] Trial 1 finished with values: [0.4059086423664789, 0.3082986317668055, 98.64634146341463] and parameters: {'explainer': 'LIME', 'threshold_percentile': 67.4880859277135, 'perturb_sigma': 2.8614830534045774, 'use_global_importance': False}.


No cached data


[I 2026-03-03 21:52:06,568] Trial 2 finished with values: [1.0, 0.012195121951219511, 170.8048780487805] and parameters: {'explainer': 'SHAP', 'threshold_percentile': 34.36417240936095, 'perturb_sigma': 0.8152775884283918, 'use_global_importance': False}.


No cached data


In [ ]:
# ========== Extracting Final Rules: Crop ==========
# INFO: Selected Trial 12 using SHAP.
# INFO: Fitting global thresholds on TRAIN set...
# /home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
#   super().__init__(**kwargs)
# /home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
#   super().__init__(**kwargs)
# INFO: Extracting final rules for 18000 TRAIN samples...
# No cached data
# WARN: Rule 3 has less than 1 selected features.
# Checkpoint saved at index: 100
# ...
# Checkpoint saved at index: 4100
# WARN: Rule 4117 has less than 1 selected features.
# WARN: Rule 4125 has less than 1 selected features.
# WARN: Rule 4146 has less than 1 selected features.
# WARN: Rule 4150 has less than 1 selected features.
# WARN: Rule 4159 has less than 1 selected features.
# Checkpoint saved at index: 4200
# WARN: Rule 4210 has less than 1 selected features.
# WARN: Rule 4233 has less than 1 selected features.
# ...
# WARN: Rule 7637 has less than 1 selected features.
# Checkpoint saved at index: 7640
# WARN: Rule 7643 has less than 1 selected features.
# Checkpoint saved at index: 7650
# ...
# Checkpoint saved at index: 11550
# WARN: Rule 11555 has less than 1 selected features.
# Checkpoint saved at index: 11560
# Checkpoint saved at index: 11570
# Checkpoint saved at index: 11580
# Checkpoint saved at index: 11590
# Checkpoint saved at index: 11600
# Checkpoint saved at index: 11610
# Checkpoint saved at index: 11620
# Checkpoint saved at index: 11630
# Checkpoint saved at index: 11640
# Checkpoint saved at index: 11650
# Checkpoint saved at index: 11660
# Checkpoint saved at index: 11670
# Checkpoint saved at index: 11680
# Checkpoint saved at index: 11690
# Checkpoint saved at index: 11700
# Checkpoint saved at index: 11710
# ...
# --- SKIPPING FordA: Target artifact 'pvts.pickle' already exists. ---
#
# ==================================================
#  STARTING TRANSACTION: FordB
# ==================================================
#
# ========== Starting Optimization: FordB ==========
# /home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
#   super().__init__(**kwargs)
# /home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
#   super().__init__(**kwargs)
# [I 2026-03-03 09:25:26,915] Using an existing study with name 'FordB' instead of creating a new one.
# INFO: Starting/Resuming study. Target: 44 more trials. Prior time spent: 3861.9s.
# No cached data


In [ ]:
# 3. Run Multivariate Loop
# 4 Hours timeout (14400s), 5% stratified pool (capped naturally by minimums in the function)
print("\n" + "#" * 50)
print(" INITIATING MULTIVARIATE PIPELINE")
print("#" * 50)

process_datasets(
    paths=multi_paths,
    timeout=4 * 60 * 60,
    pool_fraction=0.10,
    is_test_run=IS_TEST_RUN,
    error_log_path="failed_datasets_multivariate.txt",
    min_trials=20,
    n_trials=60,
    db_path="sqlite:///.optuna_phar_multivar.sqlite3",
)

print("\nGLOBAL PIPELINE EXECUTION COMPLETED.")

## 6. Execution Summary & Balancing Report
Final diagnostic output confirming the total number of processed datasets, serialization paths, and any skipped iterations, providing a clean baseline for potential parallel load balancing.

In [ ]:
time.time()